# CCC - Constant Conditional Correlation (Solution)

**Referencia**: Bollerslev, T. (1990). *Modelling the Coherence in Short-Run Nominal Exchange Rates: A Multivariate Generalized ARCH Model*. Review of Economics and Statistics, 72(3), 498-505.

---

O modelo CCC (Constant Conditional Correlation) e o modelo multivariado GARCH mais simples.
Ele assume que as **correlacoes entre os ativos sao constantes** ao longo do tempo, enquanto
as **volatilidades individuais** variam segundo processos GARCH univariados.

### Estrutura do modelo

A matriz de covariancia condicional e decomposta como:

$$H_t = D_t \, R \, D_t$$

onde:
- $D_t = \text{diag}(\sigma_{1,t}, \sigma_{2,t}, \ldots, \sigma_{k,t})$ e a matriz diagonal de volatilidades condicionais
- $R$ e a matriz de correlacao **constante** (estimada a partir dos residuos padronizados)
- Cada $\sigma_{i,t}^2$ segue um processo GARCH(1,1) univariado

### Neste notebook

1. Analise exploratoria multivariada
2. Estimacao do modelo CCC
3. Volatilidades condicionais individuais
4. Matriz de covariancia condicional
5. Limitacoes do CCC

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Adicionar utils ao path
sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_conditional_covariance, plot_correlation_heatmap

from archbox.multivariate import CCC

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

# Carregar dados fx_majors (4 series de retornos de cambio)
data_path = os.path.join("..", "data", "fx_majors.csv")
df = pd.read_csv(data_path, parse_dates=["date"], index_col="date")

print(f"Dataset: {df.shape[0]} observacoes, {df.shape[1]} series")
print(f"Series: {list(df.columns)}")
print(f"Periodo: {df.index[0].date()} a {df.index[-1].date()}")
df.head()

## 1. Analise exploratoria multivariada

Antes de estimar qualquer modelo multivariado, precisamos entender:
- A **estrutura de correlacao** entre as series
- As **volatilidades** individuais e seus padroes
- Se ha evidencias de **clustering de volatilidade** conjunto

In [ ]:
# Calcule a matriz de correlacao incondicional e plote heatmap

# Estatisticas descritivas
print("Estatisticas descritivas dos retornos:")
print(df.describe().round(6))
print()

# Matriz de correlacao incondicional
corr_matrix = df.corr().values
labels = [col.upper() for col in df.columns]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap de correlacao
plot_correlation_heatmap(corr_matrix, labels, title="Correlacao Incondicional - FX Majors", ax=axes[0])

# Retornos ao longo do tempo
for col in df.columns:
    axes[1].plot(df.index, df[col], alpha=0.6, linewidth=0.5, label=col.upper())
axes[1].set_title("Retornos Diarios - FX Majors")
axes[1].set_xlabel("Data")
axes[1].set_ylabel("Retorno")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 2. O modelo CCC

O CCC decompoe a covariancia condicional em duas partes:

$$H_t = D_t \, R \, D_t$$

**Estimacao em duas etapas:**

1. **Etapa 1**: Estimar GARCH(1,1) univariado para cada serie $i$:
$$\sigma_{i,t}^2 = \omega_i + \alpha_i \, \varepsilon_{i,t-1}^2 + \beta_i \, \sigma_{i,t-1}^2$$

2. **Etapa 2**: Calcular residuos padronizados $z_{i,t} = \varepsilon_{i,t} / \sigma_{i,t}$ e estimar:
$$R = \frac{1}{T} \sum_{t=1}^{T} z_t \, z_t'$$

A vantagem e que a estimacao e **computacionalmente simples** - nao requer otimizacao conjunta
de todos os parametros.

In [ ]:
# Estime CCC com archbox

# Preparar dados como array numpy
returns = df.values

# Estimar modelo CCC-GARCH(1,1)
model_ccc = CCC(returns, univariate_model="GARCH", univariate_order=(1, 1))
results_ccc = model_ccc.fit(method="two_step", disp=True)

# Resultados
print(f"\nLog-likelihood: {results_ccc.loglike:.4f}")
print(f"AIC: {results_ccc.aic:.4f}")
print(f"BIC: {results_ccc.bic:.4f}")
print(f"N observacoes: {results_ccc.n_obs}")
print(f"N series: {results_ccc.n_series}")

# Matriz de correlacao constante estimada
R_ccc = results_ccc.dynamic_correlation[0]  # constante, pegar t=0
print("\nMatriz de correlacao constante R:")
print(pd.DataFrame(R_ccc, index=labels, columns=labels).round(4))

## 3. Volatilidades individuais

No CCC, cada serie segue um GARCH univariado independente. A matriz $D_t$ contem
as volatilidades condicionais na diagonal:

$$D_t = \begin{pmatrix} \sigma_{1,t} & 0 & \cdots & 0 \\ 0 & \sigma_{2,t} & \cdots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \cdots & \sigma_{k,t} \end{pmatrix}$$

Estas volatilidades capturam o **clustering de volatilidade** individual de cada serie.

In [ ]:
# Plote as volatilidades condicionais individuais

cond_vol = results_ccc.conditional_volatility  # shape: (T, k)
dates = df.index

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
axes = axes.ravel()

for i, (col, label) in enumerate(zip(df.columns, labels, strict=False)):
    ax = axes[i]
    ax.plot(dates, cond_vol[:, i], color="steelblue", linewidth=0.8, label=f"$\\sigma_{{{label},t}}$")
    ax.fill_between(dates, 0, cond_vol[:, i], alpha=0.2, color="steelblue")
    ax.set_title(f"Volatilidade Condicional - {label}")
    ax.set_ylabel("$\\sigma_t$")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Data")
axes[-2].set_xlabel("Data")
fig.suptitle("Volatilidades Condicionais Individuais (CCC-GARCH)", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 4. Matriz de covariancia condicional

A matriz de covariancia condicional $H_t$ combina as volatilidades individuais com a
correlacao constante:

$$H_t = D_t \, R \, D_t$$

Cada elemento:
- **Diagonal**: $h_{ii,t} = \sigma_{i,t}^2$ (variancia condicional)
- **Fora da diagonal**: $h_{ij,t} = \rho_{ij} \, \sigma_{i,t} \, \sigma_{j,t}$ (covariancia condicional)

Note que as covariancias variam no tempo (via $\sigma_{i,t}$), mesmo com correlacoes constantes.

In [ ]:
# Extraia e visualize H_t para datas especificas

H_t = results_ccc.dynamic_covariance  # shape: (T, k, k)

# Selecionar datas especificas para visualizar H_t
sample_indices = [0, len(dates)//4, len(dates)//2, 3*len(dates)//4, -1]

fig, axes = plt.subplots(1, len(sample_indices), figsize=(20, 4))
for idx, t in enumerate(sample_indices):
    plot_correlation_heatmap(
        H_t[t], labels,
        title=f"$H_t$ em {dates[t].strftime('%Y-%m-%d')}",
        ax=axes[idx], vmin=None, vmax=None
    )
fig.suptitle("Matriz de Covariancia Condicional em Datas Selecionadas", fontsize=14, y=1.05)
fig.tight_layout()
plt.show()

# Plotar covariancias condicionais ao longo do tempo
cov_dict = {}
for i in range(len(labels)):
    cov_dict[f"Var({labels[i]})"] = H_t[:, i, i]
    for j in range(i+1, len(labels)):
        cov_dict[f"Cov({labels[i]},{labels[j]})"] = H_t[:, i, j]

plot_conditional_covariance(dates, cov_dict, title="CCC - Covariancias Condicionais")
plt.show()

## 5. Limitacoes do CCC

A principal limitacao do CCC e a hipotese de **correlacao constante**. Na pratica,
correlacoes entre ativos financeiros tendem a:

- **Aumentar em crises** (contagio financeiro)
- **Variar com o regime economico** (expansao vs. recessao)
- **Apresentar persistencia** (mudancas lentas ao longo do tempo)

Podemos testar esta hipotese comparando a correlacao constante estimada pelo CCC
com **correlacoes rolling** (janela movel).

In [ ]:
# Calcule correlacoes rolling (janela 250 dias) e compare com CCC

window = 250  # 1 ano de trading days

# Calcular correlacoes rolling para todos os pares
pairs = []
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        pairs.append((i, j, f"{labels[i]}-{labels[j]}"))

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
axes = axes.ravel()

for idx, (i, j, pair_name) in enumerate(pairs):
    # Correlacao rolling
    rolling_corr = df.iloc[:, i].rolling(window).corr(df.iloc[:, j])

    ax = axes[idx]
    ax.plot(dates, rolling_corr, color="steelblue", linewidth=0.8, label=f"Rolling ({window}d)")
    ax.axhline(y=R_ccc[i, j], color="red", linestyle="--", linewidth=1.5,
               label=f"CCC: {R_ccc[i, j]:.3f}")
    ax.set_title(f"Correlacao: {pair_name}")
    ax.set_ylabel("Correlacao")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Correlacoes Rolling vs. CCC Constante", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print("Se as correlacoes rolling variam significativamente ao redor da constante CCC,")
print("isso sugere que um modelo com correlacoes dinamicas (como DCC) seria mais adequado.")